## Wstęp do Uczenia Maszynowego

#### Laboratorium 06

### Zakres laboratorium

- regresja logistyczna: model, interpretacja współczynników, predykcja
- regularyzacja L1 i L2 w regresji logistycznej
- znaczenie skalowania zmiennych dla regularyzacji (z wykorzystaniem `Pipeline` i `ColumnTransformer` z lab05)

### 1. Regresja logistyczna

W modelu regresji liniowej predykcja jest ciągłą wartością (np. cena domu, wzrost):

$$
\hat{y} = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p.
$$

Ponieważ wartości prognozowane przez model regresji liniowej są rzeczywiste (od $-\infty$ do $+\infty$), a naszym celem jest klasyfikacja, stosujemy funkcję logistyczną jako przekształcenie:

$$
\sigma(z) = \frac{1}{1 + \mathrm{e}^{-z}} = \frac{\mathrm{e}^{z}}{1 + \mathrm{e}^{z}},
$$

która przyjmuje wartości z przedziału $(0, 1)$. Podstawiając za $z$ kombinację liniową cech, otrzymujemy model regresji logistycznej, który zwraca prawdopodobieństwo $P(Y=1 \mid X)$:

$$
p = P(Y=1 \mid X) = \frac{1}{1 + \mathrm{e}^{-(\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p)}} = \frac{\mathrm{e}^{\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p}}{1 + \mathrm{e}^{\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p}}.
$$

Równoważnie, logarytm szans (*logit*) jest równy kombinacji liniowej cech:

$$
\ln\left(\frac{p}{1 - p}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p.
$$

Stąd nazwa modelu: regresję liniową wykonujemy na logarytmie szans.

**Interpretacja współczynników.** Wzrost $x_j$ o 1 (przy ustalonych pozostałych cechach) zwiększa logarytm szans o $\beta_j$, czyli mnoży szanse $\frac{p}{1-p}$ przez $\mathrm{e}^{\beta_j}$ (iloraz szans, *odds ratio*).

**Dopasowanie modelu.** Współczynniki wyznaczamy, minimalizując ujemną log-wiarogodność (*log-loss*):

$$
\mathcal{L}(\boldsymbol{\beta}) = -\sum_{i=1}^{n}\left[y_i\log(p_i) + (1-y_i)\log(1-p_i)\right].
$$

> **Uwaga.** `LogisticRegression()` domyślnie stosuje regularyzację L2 z `C=1.0` (patrz sekcja 2). Aby dopasować model bez regularyzacji, ustawiamy `C=np.inf`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from textwrap import fill

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

-----
##### *Zadanie 1*

Dane `SAheart.data` dotyczą występowania choroby wieńcowej (zmienna `chd`) u mężczyzn z regionu wysokiego ryzyka w RPA. Zmienne objaśniające to m.in. ciśnienie skurczowe (`sbp`), spożycie tytoniu (`tobacco`), cholesterol LDL (`ldl`), występowanie choroby w rodzinie (`famhist`) i wiek (`age`).

a) Dopasuj model regresji logistycznej bez regularyzacji na zbiorze treningowym. Oceń go na zbiorze testowym za pomocą ROC AUC.

b) Zinterpretuj współczynniki przy zmiennych `age` i `famhist`.

c) Dla obserwacji `new_obs` oblicz prawdopodobieństwo `chd = 1` **ręcznie** — na podstawie wyznaczonych współczynników i wzoru na $p$, bez użycia `predict_proba`. Porównaj wynik z `predict_proba`.

> Źródło danych: https://hastie.su.domains/ElemStatLearn/ (*South African Heart Disease*).

In [ ]:
SAheart = pd.read_csv('SAheart.data')

# identyfikator nie jest cechą; kodujemy zmienną `famhist` na zmienną binarną
y = SAheart['chd']
X = SAheart.drop(columns=['chd', 'row.names'])
X['famhist'] = X['famhist'].map({'Present': 1, 'Absent': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
X_train.head()

In [ ]:
# c) nowa obserwacja
new_obs = pd.DataFrame({
    'sbp': [127], 'tobacco': [0], 'ldl': [4.44], 'adiposity': [18], 'famhist': [1],
    'typea': [49], 'obesity': [25], 'alcohol': [0], 'age': [30],
})

### 2. Regularyzacja regresji logistycznej

Model z wieloma cechami lub dopasowany na niewielkiej liczbie obserwacji łatwo się przeucza: dopasowuje się do szumu w danych treningowych, co zwykle objawia się dużymi wartościami współczynników i dużą wariancją estymatorów. Regularyzacja ogranicza wielkość współczynników — wprowadza niewielkie obciążenie, ale zmniejsza wariancję i często poprawia jakość predykcji na nowych danych.

Do funkcji straty dodajemy karę za duże współczynniki:

$$
\mathcal{L}_\lambda(\boldsymbol{\beta}) = \mathcal{L}(\boldsymbol{\beta}) + \lambda P(\boldsymbol{\beta}),
$$

gdzie:
- dla kary L2: $P(\boldsymbol{\beta}) = \frac{1}{2}\|\boldsymbol{\beta}\|_2^2 = \frac{1}{2}\sum_{j=1}^{p}\beta_j^2$,
- dla kary L1: $P(\boldsymbol{\beta}) = \|\boldsymbol{\beta}\|_1 = \sum_{j=1}^{p}|\beta_j|$.

Wyraz wolny $\beta_0$ zwykle nie podlega karze. L2 zmniejsza wartości współczynników, zaś L1 może wyzerować część z nich (selekcja zmiennych).

W `scikit-learn` parametr `C` jest odwrotnością siły regularyzacji:

$$
C = \frac{1}{\lambda},
$$

więc **mniejsze `C` oznacza silniejszą regularyzację**. 


Rodzaj kary wybieramy parametrem `l1_ratio`: 
- `l1_ratio=0` to kara L2,
- `l1_ratio=1` to kara L1.

W starszych wersjach `scikit-learn` także: `penalty='l2'` / `penalty='l1'`. 

Karę L1 obsługują m.in. solvery `saga` i `liblinear` ([dokumentacja](https://scikit-learn.org/stable/modules/linear_model.html#solvers)). Dla kary L1 będziemy używać `saga`, z dwoma zastrzeżeniami:
- **wyrazu wolnego nie karzemy** — odpowiada on bazowemu logarytmowi szans, a nie złożoności modelu. `saga` go nie regularyzuje, natomiast `liblinear` traktuje wyraz wolny jak dodatkową stałą cechę i nakłada na niego karę. Wpływ tej kary można zmniejszyć, zwiększając parametr `intercept_scaling`;
- **`saga` szybko zbiega tylko dla cech o podobnej skali** — na danych nieskalowanych może nie osiągnąć rozwiązania.

-----
##### *Zadanie 2*

Eksperyment symulacyjny. Wygenerujmy dane w następujący sposób: 
- Niech $n = 200$, $p = 5$.
- $\beta_0 = 1$, $\beta = [2, 1, 0.5, 0.01, 0]$. 
- $x_i \sim N_p(0, I)$ dla $i = 1,2, \dots, n$. 
- $y_i \sim Bern(p_i)$, gdzie $p_i=\frac{\mathrm{e}^{\beta_0 + x_{i\cdot}\beta}}{1 + \mathrm{e}^{\beta_0 + x_{i\cdot}\beta}}$.

a) Dopasuj model regresji logistycznej bez regularyzacji. Porównaj prawdziwe wartości wektora $\beta$ z wyestymowanymi.

b) Dopasuj model regresji logistycznej z regularyzacją $L2$ i wartościami $C = (10, 5, 2, 1, 0.5, 0.1, 0.01, 0.005)$. Jak zmieniają się współczynniki wraz ze wzrostem $\lambda$?

c) Dopasuj model regresji logistycznej z regularyzacją $L1$ i wartościami $C = (10, 5, 2, 1, 0.5, 0.1, 0.01, 0.005)$. Jak zmieniają się współczynniki wraz ze wzrostem $\lambda$?

d) Powtórz eksperyment 500 razy dla modelu logistycznego bez regularyzacji, z regularyzacją $L2$ i parametrem $C=1$ oraz z regularyzacją $L1$ i parametrem $C=1$. Oblicz błąd średniokwadratowy estymatorów współczynników (łącznie z wyrazem wolnym):

$$
\text{MSE} = \frac{1}{K}\sum_{k=1}^{K}\left[(\hat{\beta}_0^{(k)} - \beta_0)^2 + \|\hat{\beta}^{(k)} - \beta\|_2^2\right].
$$

### 3. Skalowanie a regularyzacja

Kara L1 lub L2 traktuje wszystkie współczynniki jednakowo, a wielkość współczynnika zależy od skali odpowiadającej mu zmiennej. Przykładowo, `credit_amount` (kwota kredytu, wartości rzędu tysięcy) będzie miała bardzo mały współczynnik, nawet jeśli jest ważną cechą. Dlatego przed regularyzacją standaryzujemy cechy numeryczne; poniżej porównamy oba warianty na tych samych danych.

Przygotowanie danych łączymy z modelem w `Pipeline`, tak jak w lab05: `ColumnTransformer` koduje cechy kategoryczne (`OneHotEncoder`) i — w wariancie ze skalowaniem — standaryzuje cechy numeryczne. Wszystkie transformacje są dopasowywane wyłącznie do danych treningowych (a w walidacji krzyżowej — do danych treningowych danego folda).

-----
##### *Zadanie 3*

Wpływ skalowania zmiennych. Korzystając z `Pipeline` i `ColumnTransformer`, dopasuj modele regresji logistycznej (`C=0.03`):
1. z regularyzacją L2 dla danych bez skalowania i dla danych ze skalowaniem cech numerycznych (`StandardScaler`);
2. z regularyzacją L1 dla danych bez skalowania i dla danych ze skalowaniem cech numerycznych (`StandardScaler`).

Porównaj współczynniki oraz ROC AUC na zbiorze testowym.

> **Wskazówka.** Dla modelu L1 ze skalowaniem użyj solvera `saga`. Na danych bez skalowania `saga` nie zbiega, dlatego tam użyj `liblinear` z `intercept_scaling=1000`, aby praktycznie wyłączyć karę na wyrazie wolnym (patrz sekcja 2).

In [ ]:
# Wczytanie danych (zbiór German Credit, źródło: OpenML `credit-g`)
df = pd.read_csv('credit-g.csv')
print(df.shape)

# wybranie kilku kolumn
sel_cols = ['age', 'existing_credits', 'duration', 'credit_amount', 'savings_status', 'credit_history', 'class']
df = df[sel_cols].copy()
df.head(10)

In [ ]:
# kodowanie klas
df['class'] = (df['class'] == 'good').astype(int)

# wybór X i y
X = df.drop(columns='class')
y = df['class']

In [ ]:
numerical_features = ['age', 'existing_credits', 'duration', 'credit_amount']
categorical_features = ['savings_status', 'credit_history']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Przygotuj dwa obiekty ColumnTransformer (z parametrem verbose_feature_names_out=False):
# - preprocessor_raw: cechy numeryczne bez zmian ('passthrough'), cechy kategoryczne -> OneHotEncoder(drop='first', handle_unknown='ignore')
# - preprocessor_scaled: cechy numeryczne -> StandardScaler(), cechy kategoryczne jak wyżej

In [ ]:
# Funkcje pomocnicze


def get_coefs(model):
    # współczynniki regresji logistycznej z nazwami cech po przygotowaniu danych
    names = model.named_steps['pre'].get_feature_names_out()
    return pd.Series(model.named_steps['clf'].coef_[0], index=names)


def test_auc(model):
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])


def plot_coefficient_comparison(coef_raw, coef_scaled, title):
    comparison = pd.DataFrame({
        'bez skalowania': coef_raw,
        'po skalowaniu': coef_scaled,
    })
    comparison = comparison.loc[comparison.abs().max(axis=1).sort_values(ascending=False).index]

    y_pos = np.arange(len(comparison))
    labels = [fill(name, width=26) for name in comparison.index]
    fig, ax = plt.subplots(figsize=(10, max(4, 0.55 * len(comparison))))
    ax.barh(y_pos - 0.2, comparison['bez skalowania'], height=0.4, label='bez skalowania')
    ax.barh(y_pos + 0.2, comparison['po skalowaniu'], height=0.4, label='po skalowaniu')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_yticks(y_pos, labels)
    ax.invert_yaxis()
    ax.set_xlabel('Wartość współczynnika β')
    ax.set_ylabel('Cecha')
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    plt.show()

1. z regularyzacją L2 dla danych bez skalowania i dla danych ze skalowaniem cech numerycznych (`StandardScaler`)

2. z regularyzacją L1 dla danych bez skalowania i dla danych ze skalowaniem cech numerycznych (`StandardScaler`)

#### Jak dobrać `C`?

W tym zadaniu wartość `C=0.03` dobraliśmy tak, aby efekt skalowania był dobrze widoczny. W praktyce `C` (oraz rodzaj kary) to hiperparametry, które dobieramy walidacją krzyżową na zbiorze treningowym — tak jak `max_depth` drzewa w lab04 i lab05. Wystarczy przekazać cały pipeline do `GridSearchCV`, np. z siatką `{'clf__C': np.logspace(-3, 2, 11)}`; skalowanie jest wtedy dopasowywane osobno w każdym foldzie. Alternatywą jest `LogisticRegressionCV`, które samo przeszukuje wartości `C` z użyciem walidacji krzyżowej.